In [ ]:
import os
import sys
sys.path.append(os.path.abspath("../.."))

from dotenv import load_dotenv
load_dotenv()  # loads QDRANT_URL / QDRANT_API_KEY etc. from the .env file in this same folder

from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore
from langchain_qdrant import QdrantVectorStore, RetrievalMode, FastEmbedSparse
from langchain_core.documents import Document
from langchain_classic.chains import HypotheticalDocumentEmbedder, LLMChain

from qdrant_client.http.models import SparseVectorParams
from qdrant_client import models

from src.backend.logger import GLOBAL_LOGGER as log
from src.backend.core.config import settings
from src.backend.rag.embeddings import get_embeddings

****Data Ingestion****

In [2]:
def load_document(directory_path):
    try:
        documents = []
        for filename in os.listdir(directory_path):
            file_path = os.path.join(directory_path, filename)
            if filename.endswith(".pdf"):
                loader = PyPDFLoader(file_path)
                documents.extend(loader.load())
            elif filename.endswith(".docx"):
                # First: extract normal docx text
                loader = Docx2txtLoader(file_path)
                documents.extend(loader.load())
            elif filename.endswith(".txt"):
                loader = TextLoader(file_path, encoding="utf-8")
                documents.extend(loader.load())

        return documents
    except Exception as e:
        log.error("Error loading documents from directory", error=str(e), directory=directory_path)
        raise e

docs = load_document("./TempData")
docs

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-14T08:45:46+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-12-14T08:45:46+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './TempData\\Acme_FY2024_UltraDense_Report.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='ACME MANUFACTURING LTD – FY2024 CONSOLIDATED REPORT\nAcme Manufacturing Ltd is a multinational industrial manufacturing company operating across the United States, Germany, and the\nUnited Kingdom. The company manufactures heavy machinery, automotive components, and precision-engineered industrial\nequipment for aerospace and defense sectors. Acme’s customers include original equipment manufacturers, government agencies,\nand industrial distributors. The company prepares consolidated financial statements in accordance with International Financial\nReporting Standards 

****Chunking****

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=50)
    split_docs = text_splitter.split_documents(documents)
    
    return split_docs

docs_splitted = split_documents(docs)

****Embeddings****

In [ ]:
embeddings = get_embeddings("OpenAI (text-embedding-3-small)")

#Check the size of the embeddings
embeddings_size = len(embeddings.embed_query("check embedding size"))
print(embeddings_size)

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


1536


****Create Qdrant Client****

In [ ]:
from qdrant_client import QdrantClient

qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")
if not (qdrant_url and qdrant_api_key):
    raise RuntimeError("QDRANT_URL / QDRANT_API_KEY not set -- check the .env file in this folder")

qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key,
)

****Create Collection****

In [6]:
#Create Collection
collection_name = "multi-document-assist"
if not qdrant_client.collection_exists(collection_name=collection_name):
    log.info("Collection does not exist, Creating Collection", collection_name=collection_name)
    qdrant_client.create_collection(collection_name, 
                                  vectors_config={
                                    "size": embeddings_size,
                                    "distance": "Cosine"
                                  }
                                  )
else:
    log.info("Collection already exists", collection_name=collection_name)

HTTP Request: GET https://c72720c9-8267-4e2b-91cc-e2ce70a87e6a.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/multi-document-assist/exists "HTTP/1.1 200 OK"
{"collection_name": "multi-document-assist", "timestamp": "2025-12-20T11:07:04.853928Z", "level": "info", "event": "Collection already exists"}


****Create Vector Store****

In [7]:
vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name=collection_name,
    embedding=embeddings,
)

HTTP Request: GET https://c72720c9-8267-4e2b-91cc-e2ce70a87e6a.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/multi-document-assist "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


****Adding Document to Qdrant Vectorstore****

In [8]:
vector_store.add_documents(documents=docs)

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: PUT https://c72720c9-8267-4e2b-91cc-e2ce70a87e6a.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/multi-document-assist/points?wait=true "HTTP/1.1 200 OK"


['ba7d684f05d74701ad0afed7b895c736', '3fe8e9a2c4eb4347a794a5b9d82c8418']

****Retrieval Phase****

In [9]:
results = vector_store.similarity_search(
    "what is the revenue increase in FY24?", k=1
)

results

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://c72720c9-8267-4e2b-91cc-e2ce70a87e6a.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/multi-document-assist/points/query "HTTP/1.1 200 OK"


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-14T08:45:46+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-12-14T08:45:46+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './TempData\\Acme_FY2024_UltraDense_Report.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', '_id': '3fe8e9a2-c4eb-4347-a794-a5b9d82c8418', '_collection_name': 'multi-document-assist'}, page_content='BALANCE SHEET, COMPLIANCE & RISK DISCLOSURES\nBALANCE SHEET SUMMARY (FY2024): Total assets amounted to USD 200.0 million, comprising current assets of USD 50.0\nmillion and non-current assets of USD 150.0 million. Total liabilities stood at USD 140.0 million, including short-term borrowings of\nUSD 32.0 million and long-term debt of USD 80.0 million. Total equity attributable to shareholders amounted to USD 60.0 million.\nBalance Sheet (USD)\nFY2024\nCash & Equivalents\n18,000,